<a href="https://colab.research.google.com/github/sanmquin/AI/blob/main/src/Graphiko/Create-Graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install pymongo if not already present
!pip install pymongo dnspython

import pymongo
from pymongo import MongoClient
from google.colab import userdata

try:
    # Retrieve the URI from Colab Secrets
    uri = userdata.get('MONGODB_URI')
    client = MongoClient(uri)

    # Send a ping to confirm a successful connection
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(f"An error occurred: {e}")
    print("\nMake sure you have added 'MONGODB_URI' to your Colab Secrets (the key icon on the left).")

Pinged your deployment. You successfully connected to MongoDB!


In [4]:
# Access Finder DB collections
db = client['finder']
clusters_col = db['ChannelDescriptions_clusters']
items_col = db['ChannelDescriptions_items']
channels_col = db['channels']

# 1) Get the latest version in ChannelDescriptions_clusters
latest = clusters_col.find_one(sort=[('version', -1), ('createdAt', -1)])
latest_version = latest['version'] if latest else None

print(f"Latest cluster version: {latest_version}")



Latest cluster version: 2


In [5]:
# 2) Get the _id of the business cluster in the latest version
if latest_version is None:
    print('No clusters found.')
    business_cluster = None
else:
    business_cluster = clusters_col.find_one({
        'version': latest_version,
        'name': {'$regex': '^business', '$options': 'i'}
    })

    if business_cluster:
        business_cluster_id = business_cluster['_id']
        print(f"Business cluster id: {business_cluster_id}")
        print(f"Business cluster name: {business_cluster.get('name')}")
    else:
        business_cluster_id = None
        print(f"No business cluster found in version {latest_version}.")



Business cluster id: 69e41878685f30ed081ec5a6
Business cluster name: Business, Venture Capital, and Entrepreneurship


In [6]:
# 3) Get channels in the business cluster (id + title)
if business_cluster_id is None:
    print('Cannot fetch channels without a business cluster id.')
else:
    item_docs = list(items_col.find(
        {'clusterId': business_cluster_id},
        {'_id': 0, 'textId': 1}
    ))
    channel_ids = [d['textId'] for d in item_docs if d.get('textId')]

    channel_docs = list(channels_col.find(
        {'channelId': {'$in': channel_ids}},
        {'_id': 0, 'channelId': 1, 'title': 1}
    ))

    channel_by_id = {c['channelId']: c for c in channel_docs}

    print(f"Channels in business cluster ({len(channel_ids)} total):")
    for cid in channel_ids:
        c = channel_by_id.get(cid)
        title = c.get('title') if c else None
        print({'id': cid, 'title': title})



Channels in business cluster (27 total):
{'id': 'UC9cn0TuPq4dnbTY-CBsm8XA', 'title': 'a16z'}
{'id': 'UCESLZhusAkFfsNsApnjF_Cg', 'title': 'All-In Podcast'}
{'id': 'UCcefcZRL2oaA_uBNeo5UOWg', 'title': 'Y Combinator'}
{'id': 'UCvxm0qTrGN_1LMYgUaftWyQ', 'title': 'Peter H. Diamandis'}
{'id': 'UCPjNBjflYl0-HQtUvOx0Ibw', 'title': 'Greg Isenberg'}
{'id': 'UC-yRDvpR99LUc5l7i7jLzew', 'title': 'Bg2 Pod'}
{'id': 'UCJEDniyP_YtcsXikBELqicw', 'title': None}
{'id': 'UCBH5VZE_Y4F3CMcPIzPEB5A', 'title': None}
{'id': 'UC1LpsuAUaKoMzzJSEt5WImw', 'title': 'Asianometry'}
{'id': 'UCznv7Vf9nBdJYvBagFdAHWw', 'title': 'Tim Ferriss'}
{'id': 'UCK-zlnUfoDHzUwXcbddtnkg', 'title': None}
{'id': 'UC6t1O76G0jYXOAoYCm153dA', 'title': "Lenny's Podcast"}
{'id': 'UCIHdDJ0tjn_3j-FS7s_X1kQ', 'title': 'Valuetainment'}
{'id': 'UCf0PBRjhf0rF8fWBIxTuoWA', 'title': '20VC with Harry Stebbings'}
{'id': 'UCWrF0oN6unbXrWsTN7RctTw', 'title': 'Sequoia Capital'}
{'id': 'UCyaN6mg5u8Cjy2ZI4ikWaug', 'title': None}
{'id': 'UCqvaXJ1K3HheTPNj

In [7]:
# Optional: inspect the business cluster document
if business_cluster_id is not None:
    import pprint
    pprint.pprint(business_cluster)


{'__v': 0,
 '_id': ObjectId('69e41878685f30ed081ec5a6'),
 'centroid': [],
 'createdAt': datetime.datetime(2026, 4, 18, 23, 49, 12, 336000),
 'description': 'Creators and firms providing insights into the world of '
                'startups, investing, and global business trends. This '
                'includes venture capital perspectives (a16z, 20VC), business '
                'growth strategy (Alex Hormozi, My First Million), historical '
                'analysis of industry leaders (Founders Podcast, Asianometry), '
                'and financial market commentary (Graham Stephan, Anthony '
                'Pompliano).',
 'name': 'Business, Venture Capital, and Entrepreneurship',
 'summary': 'Media dedicated to business intelligence, financial markets, '
            'startup growth, and the methodologies of successful '
            'entrepreneurs.',
 'version': 2}


## Subscriptions analysis for business-cluster channels
Counts subscriptions per channel and builds an owner-overlap matrix for channel pairs.

### Persistent artifact path (Google Drive)
The common subscription matrix is persisted to this shared path:
- `/content/drive/MyDrive/finder_artifacts/common_subscription_matrix/owner_overlap_matrix.csv`
- `/content/drive/MyDrive/finder_artifacts/common_subscription_matrix/owner_overlap_matrix.pkl`
- `/content/drive/MyDrive/finder_artifacts/common_subscription_matrix/subscriptions_per_channel.csv`

This path can be reused by other notebooks (including notebooks from other repositories) as long as they mount the same Google Drive.


In [8]:
# 4) Query subscriptions for all business-cluster channels and count rows per channel
from collections import defaultdict

subscriptions_col = db['subscriptions']

if business_cluster_id is None:
    print('Cannot analyze subscriptions without a business cluster id.')
else:
    # IMPORTANT: canonical schema is documented in Finder/db-schemas.md.
    # Use subscriberChannelId as the subscription owner (author) field to avoid schema guessing bugs.
    owner_field = 'subscriberChannelId'

    # Keep only unique channel ids while preserving order
    channel_ids_unique = list(dict.fromkeys(channel_ids))

    # Pull relevant subscription rows once, then aggregate in Python
    subs_docs = list(subscriptions_col.find(
        {'channelId': {'$in': channel_ids_unique}},
        {
            '_id': 0,
            'channelId': 1,
            owner_field: 1,
        }
    ))

    print(f"Owner field used for author/subscription owner: {owner_field}")

    # 1+2) Count subscriptions per channel (replicated for all channels)
    subscriptions_per_channel = defaultdict(int)
    owners_per_channel = defaultdict(set)

    for doc in subs_docs:
        cid = doc.get('channelId')
        owner = doc.get(owner_field)
        if cid is None:
            continue
        subscriptions_per_channel[cid] += 1
        if owner:
            owners_per_channel[cid].add(owner)

    print('\nSubscription count by channel id:')
    for cid in channel_ids_unique:
        title = channel_by_id.get(cid, {}).get('title')
        print(f"- {cid} | {title} => {subscriptions_per_channel[cid]} subscriptions")


Owner field used for author/subscription owner: subscriberChannelId

Subscription count by channel id:
- UC9cn0TuPq4dnbTY-CBsm8XA | a16z => 42 subscriptions
- UCESLZhusAkFfsNsApnjF_Cg | All-In Podcast => 41 subscriptions
- UCcefcZRL2oaA_uBNeo5UOWg | Y Combinator => 46 subscriptions
- UCvxm0qTrGN_1LMYgUaftWyQ | Peter H. Diamandis => 24 subscriptions
- UCPjNBjflYl0-HQtUvOx0Ibw | Greg Isenberg => 22 subscriptions
- UC-yRDvpR99LUc5l7i7jLzew | Bg2 Pod => 16 subscriptions
- UCJEDniyP_YtcsXikBELqicw | None => 14 subscriptions
- UCBH5VZE_Y4F3CMcPIzPEB5A | None => 15 subscriptions
- UC1LpsuAUaKoMzzJSEt5WImw | Asianometry => 65 subscriptions
- UCznv7Vf9nBdJYvBagFdAHWw | Tim Ferriss => 17 subscriptions
- UCK-zlnUfoDHzUwXcbddtnkg | None => 14 subscriptions
- UC6t1O76G0jYXOAoYCm153dA | Lenny's Podcast => 14 subscriptions
- UCIHdDJ0tjn_3j-FS7s_X1kQ | Valuetainment => 21 subscriptions
- UCf0PBRjhf0rF8fWBIxTuoWA | 20VC with Harry Stebbings => 13 subscriptions
- UCWrF0oN6unbXrWsTN7RctTw | Sequoia Capit

In [9]:
# 3+4+5) Build pairwise owner overlap matrix and persist using the common adjacency schema
import json
import pandas as pd
from pathlib import Path

if business_cluster_id is None:
    print('Skipping matrix creation (no business cluster id).')
else:
    # Common schema constants
    GRAPH_SCHEMA_NAME = 'graphiko.adjacency'
    GRAPH_SCHEMA_VERSION = '1.0.0'
    GRAPH_KIND = 'subscriptions_owner_overlap'

    node_records = []
    for cid in channel_ids_unique:
        title = channel_by_id.get(cid, {}).get('title') or 'Unknown title'
        node_records.append({'node_id': str(cid), 'node_label': title})

    nodes_df = pd.DataFrame(node_records)

    ordered_ids = nodes_df['node_id'].tolist()
    ordered_labels = [f"{r['node_label']} ({r['node_id']})" for r in node_records]

    # Matrix[i][j] = number of unique owners subscribed to both channels i and j
    matrix = []
    for cid_i in ordered_ids:
        row = []
        owners_i = owners_per_channel.get(cid_i, set())
        for cid_j in ordered_ids:
            owners_j = owners_per_channel.get(cid_j, set())
            row.append(len(owners_i & owners_j))
        matrix.append(row)

    overlap_df = pd.DataFrame(matrix, index=ordered_ids, columns=ordered_ids)

    print('Owner overlap matrix (node_id x node_id):')
    print(overlap_df)

    # Persist in shared Google Drive so artifacts can be reused by notebooks across repositories.
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        raise RuntimeError('Google Drive mount is required to persist shared artifacts.') from e

    root_dir = Path('/content/drive/MyDrive/Graphiko/graphs/subscriptions_owner_overlap')
    latest_dir = root_dir / 'latest'
    versioned_dir = root_dir / f'v{GRAPH_SCHEMA_VERSION}'
    latest_dir.mkdir(parents=True, exist_ok=True)
    versioned_dir.mkdir(parents=True, exist_ok=True)

    schema_metadata = {
        'schema_name': GRAPH_SCHEMA_NAME,
        'schema_version': GRAPH_SCHEMA_VERSION,
        'graph_kind': GRAPH_KIND,
        'is_directed': False,
        'weight_semantics': 'shared_subscriber_owner_count',
        'node_count': len(ordered_ids),
        'row_node_ids': ordered_ids,
        'column_node_ids': ordered_ids,
        'row_column_labels': ordered_labels,
        'derivation': {
            'source_db': 'finder',
            'source_collections': ['ChannelDescriptions_clusters', 'ChannelDescriptions_items', 'channels', 'subscriptions'],
            'cluster_filter': 'latest version + name starts with business',
            'owner_field': owner_field
        }
    }

    nodes_path_latest = latest_dir / 'nodes.csv'
    matrix_path_latest = latest_dir / 'adjacency_matrix.csv'
    meta_path_latest = latest_dir / 'metadata.json'

    nodes_path_versioned = versioned_dir / 'nodes.csv'
    matrix_path_versioned = versioned_dir / 'adjacency_matrix.csv'
    meta_path_versioned = versioned_dir / 'metadata.json'

    nodes_df.to_csv(nodes_path_latest, index=False)
    overlap_df.to_csv(matrix_path_latest, index_label='row_node_id')
    meta_path_latest.write_text(json.dumps(schema_metadata, indent=2))

    nodes_df.to_csv(nodes_path_versioned, index=False)
    overlap_df.to_csv(matrix_path_versioned, index_label='row_node_id')
    meta_path_versioned.write_text(json.dumps(schema_metadata, indent=2))

    # Keep channel count side artifact for diagnostics
    counts_df = pd.DataFrame([
        {
            'channelId': cid,
            'title': channel_by_id.get(cid, {}).get('title'),
            'subscriptions': subscriptions_per_channel.get(cid, 0),
            'unique_owners': len(owners_per_channel.get(cid, set()))
        }
        for cid in ordered_ids
    ])
    counts_df.to_csv(latest_dir / 'subscriptions_per_channel.csv', index=False)

    print(f'Graph schema: {GRAPH_SCHEMA_NAME}@{GRAPH_SCHEMA_VERSION}')
    print(f'Saved latest nodes: {nodes_path_latest}')
    print(f'Saved latest matrix: {matrix_path_latest}')
    print(f'Saved latest metadata: {meta_path_latest}')
    print(f'Saved versioned nodes: {nodes_path_versioned}')
    print(f'Saved versioned matrix: {matrix_path_versioned}')
    print(f'Saved versioned metadata: {meta_path_versioned}')



Author overlap matrix (pairwise shared subscription owners):
                                                    a16z (UC9cn0TuPq4dnbTY-CBsm8XA)  \
a16z (UC9cn0TuPq4dnbTY-CBsm8XA)                                                  42   
All-In Podcast (UCESLZhusAkFfsNsApnjF_Cg)                                        31   
Y Combinator (UCcefcZRL2oaA_uBNeo5UOWg)                                          20   
Peter H. Diamandis (UCvxm0qTrGN_1LMYgUaftWyQ)                                    14   
Greg Isenberg (UCPjNBjflYl0-HQtUvOx0Ibw)                                         13   
Bg2 Pod (UC-yRDvpR99LUc5l7i7jLzew)                                               14   
Unknown title (UCJEDniyP_YtcsXikBELqicw)                                         14   
Unknown title (UCBH5VZE_Y4F3CMcPIzPEB5A)                                         10   
Asianometry (UC1LpsuAUaKoMzzJSEt5WImw)                                           11   
Tim Ferriss (UCznv7Vf9nBdJYvBagFdAHWw)                               

In [10]:
# 6) Build a normalized distance graph from the shared-owner adjacency matrix and persist with the same schema
import json
import numpy as np
import pandas as pd

if business_cluster_id is None:
    print('Skipping normalized-distance graph creation (no business cluster id).')
else:
    GRAPH_SCHEMA_NAME = 'graphiko.adjacency'
    GRAPH_SCHEMA_VERSION = '1.0.0'
    GRAPH_KIND = 'subscriptions_normalized_inverse_overlap_distance'

    root_dir = Path('/content/drive/MyDrive/Graphiko/graphs/subscriptions_normalized_distance')
    latest_dir = root_dir / 'latest'
    versioned_dir = root_dir / f'v{GRAPH_SCHEMA_VERSION}'
    latest_dir.mkdir(parents=True, exist_ok=True)
    versioned_dir.mkdir(parents=True, exist_ok=True)

    # Reuse in-memory matrix if available; otherwise load the persisted artifact.
    if 'overlap_df' not in globals():
        overlap_df = pd.read_csv(
            '/content/drive/MyDrive/Graphiko/graphs/subscriptions_owner_overlap/latest/adjacency_matrix.csv',
            index_col='row_node_id'
        )

    overlap_values = overlap_df.values.astype(float)

    # Convert overlap similarity into distance using inverse overlap (+1 smoothing).
    # Larger overlap => smaller distance.
    distance_raw = 1.0 / (overlap_values + 1.0)
    np.fill_diagonal(distance_raw, 0.0)

    # Row-wise normalize so total outgoing distance per node sums to 1.
    row_sums = distance_raw.sum(axis=1, keepdims=True)
    distance_norm = np.divide(
        distance_raw,
        row_sums,
        out=np.zeros_like(distance_raw),
        where=row_sums != 0
    )

    distance_df = pd.DataFrame(distance_norm, index=overlap_df.index, columns=overlap_df.columns)

    # Node labels if available
    node_ids = distance_df.index.astype(str).tolist()
    if 'nodes_df' in globals() and {'node_id', 'node_label'}.issubset(set(nodes_df.columns)):
        node_records = nodes_df[['node_id', 'node_label']].copy()
    else:
        node_records = pd.DataFrame({'node_id': node_ids, 'node_label': node_ids})

    schema_metadata = {
        'schema_name': GRAPH_SCHEMA_NAME,
        'schema_version': GRAPH_SCHEMA_VERSION,
        'graph_kind': GRAPH_KIND,
        'is_directed': True,
        'weight_semantics': 'row_normalized_inverse_shared_owner_overlap',
        'node_count': len(node_ids),
        'row_node_ids': node_ids,
        'column_node_ids': node_ids,
        'derivation': {
            'derived_from': '/content/drive/MyDrive/Graphiko/graphs/subscriptions_owner_overlap/latest/adjacency_matrix.csv',
            'transform': 'distance_raw=1/(overlap+1), then row normalization'
        }
    }

    for target_dir in [latest_dir, versioned_dir]:
        node_records.to_csv(target_dir / 'nodes.csv', index=False)
        distance_df.to_csv(target_dir / 'adjacency_matrix.csv', index_label='row_node_id')
        (target_dir / 'metadata.json').write_text(json.dumps(schema_metadata, indent=2))

    print(f'Graph schema: {GRAPH_SCHEMA_NAME}@{GRAPH_SCHEMA_VERSION}')
    print(f'Saved latest normalized matrix: {latest_dir / "adjacency_matrix.csv"}')
    print(f'Saved versioned normalized matrix: {versioned_dir / "adjacency_matrix.csv"}')



Normalized distance matrix (rows sum to ~1):
                                               a16z (UC9cn0TuPq4dnbTY-CBsm8XA)  \
a16z (UC9cn0TuPq4dnbTY-CBsm8XA)                                       0.000000   
All-In Podcast (UCESLZhusAkFfsNsApnjF_Cg)                             0.013434   
Y Combinator (UCcefcZRL2oaA_uBNeo5UOWg)                               0.015936   
Peter H. Diamandis (UCvxm0qTrGN_1LMYgUaftWyQ)                         0.015726   
Greg Isenberg (UCPjNBjflYl0-HQtUvOx0Ibw)                              0.017547   

                                               All-In Podcast (UCESLZhusAkFfsNsApnjF_Cg)  \
a16z (UC9cn0TuPq4dnbTY-CBsm8XA)                                                 0.013782   
All-In Podcast (UCESLZhusAkFfsNsApnjF_Cg)                                       0.000000   
Y Combinator (UCcefcZRL2oaA_uBNeo5UOWg)                                         0.016733   
Peter H. Diamandis (UCvxm0qTrGN_1LMYgUaftWyQ)                                   0.019658   
Gr